In [1]:
!python --version

Python 3.13.15


In [2]:
!apt-get -qq update
!apt-get -qq install unrar

W: https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2404/x86_64/InRelease: Key is stored in legacy trusted.gpg keyring (/etc/apt/trusted.gpg), see the DEPRECATION section in apt-key(8) for details.
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [3]:
from pathlib import Path

rar_path = Path("/content/tutorial.rar")
extract_path = Path("/content/tutorial")

extract_path.mkdir(exist_ok=True)

!unrar x -o+ /content/tutorial.rar /content/tutorial/

print("Extraction complete!")

files = list(extract_path.rglob("*.html"))

print("HTML files found:", len(files))

for f in files[:20]:
    print("-", f)


UNRAR 7.00 freeware      Copyright (c) 1993-2024 Alexander Roshal


Extracting from /content/tutorial.rar

Creating    /content/tutorial/tutorial                                OK
Extracting  /content/tutorial/tutorial/appendix.html                       3%  OK 
Extracting  /content/tutorial/tutorial/appetite.html                       6%  OK 
Extracting  /content/tutorial/tutorial/classes.html                       17%  OK 
Extracting  /content/tutorial/tutorial/controlflow.html                   28%  OK 
Extracting  /content/tutorial/tutorial/datastructures.html                37%  OK 
Extracting  /content/tutorial/tutorial/errors.html                        43%  OK 
Extracting  /content/tutorial/tutorial/floatingpoint.html                 48%  OK 
Extracting  /content/tutorial/tutorial/index.html                         52%  OK 
Extracting  /content/tutorial/tutorial/inputoutput.html                   59%

In [4]:
# Install the libraries required for loading and processing HTML documents.
!pip install -q beautifulsoup4 lxml pandas

In [5]:
# Import the required libraries for file handling, HTML parsing, and data organization.
from pathlib import Path
from bs4 import BeautifulSoup
import pandas as pd


# Define the path to the extracted Python Tutorial HTML files.
TUTORIAL_DIR = Path("/content/tutorial/tutorial")


# Convert an HTML file into clean plain text.
def html_to_text(path):
    # Read the HTML file using UTF-8 encoding.
    html = path.read_text(encoding="utf-8", errors="ignore")

    # Parse the HTML document using BeautifulSoup.
    soup = BeautifulSoup(html, "lxml")

    # Remove elements that are not useful for retrieval.
    for tag in soup(["script", "style", "nav"]):
        tag.decompose()

    # Extract readable text while removing unnecessary whitespace.
    text = soup.get_text("\n", strip=True)

    # Remove empty lines and extra spaces.
    lines = [line.strip() for line in text.splitlines() if line.strip()]

    # Return the cleaned document text.
    return "\n".join(lines)


# Create a list to store all processed documents.
documents = []


# Process every HTML file in the tutorial directory.
for path in sorted(TUTORIAL_DIR.glob("*.html")):
    # Extract and clean the text from the HTML file.
    text = html_to_text(path)

    # Store the text together with its source filename and length.
    documents.append({
        "source": path.name,
        "text": text,
        "length": len(text)
    })


# Convert the processed documents into a Pandas DataFrame for inspection.
df = pd.DataFrame(documents)


# Display the number of processed documents and a preview.
print("Number of documents:", len(df))
df.head()

Number of documents: 17


,source,text,length
0,appendix.html,16. Appendix — Python 3.10.21 documentation\nN...,5225
1,appetite.html,1. Whetting Your Appetite — Python 3.10.21 doc...,5329
2,classes.html,9. Classes — Python 3.10.21 documentation\nNav...,35320
3,controlflow.html,4. More Control Flow Tools — Python 3.10.21 do...,36374
4,datastructures.html,5. Data Structures — Python 3.10.21 documentat...,23755


In [6]:
# Calculate and display basic statistics about the collected documents.
print("Total characters:", df["length"].sum())
print("Average characters per document:", round(df["length"].mean()))


# Display all document sources and their text lengths.
display(
    df[["source", "length"]]
    .sort_values("length", ascending=False)
)

Total characters: 249210
Average characters per document: 14659


,source,length
3,controlflow.html,36374
2,classes.html,35320
4,datastructures.html,23755
12,modules.html,23517
8,inputoutput.html,19291
11,introduction.html,17746
5,errors.html,16159
14,stdlib2.html,15019
6,floatingpoint.html,11672
13,stdlib.html,11376


In [7]:
# Display a sample of the first document to verify that the HTML was cleaned correctly.
print("SOURCE:", df.iloc[0]["source"])
print()

# Print the first 3000 characters of the selected document.
print(df.iloc[0]["text"][:3000])

SOURCE: appendix.html

16. Appendix — Python 3.10.21 documentation
Navigation
index
modules
|
next
|
previous
|
Python
»
3.10.21 Documentation
»
The Python Tutorial
»
16.
Appendix
|
Theme
Auto
Light
Dark
|
16.
Appendix
¶
16.1.
Interactive Mode
¶
16.1.1.
Error Handling
¶
When an error occurs, the interpreter prints an error message and a stack trace.
In interactive mode, it then returns to the primary prompt; when input came from
a file, it exits with a nonzero exit status after printing the stack trace.
(Exceptions handled by an
except
clause in a
try
statement
are not errors in this context.)  Some errors are unconditionally fatal and
cause an exit with a nonzero exit; this applies to internal inconsistencies and
some cases of running out of memory.  All error messages are written to the
standard error stream; normal output from executed commands is written to
standard output.
Typing the interrupt character (usually
Control
-
C
or
Delete
) to the primary or
secondary prompt cancels th

In [10]:
# Import the required libraries for HTML parsing and text cleaning.
from pathlib import Path
from bs4 import BeautifulSoup
import pandas as pd
import re


# Define the directory containing the Python Tutorial HTML files.
TUTORIAL_DIR = Path("/content/tutorial/tutorial")


# Convert an HTML document into clean text suitable for RAG.
def html_to_text(path):
    # Read the HTML file.
    html = path.read_text(encoding="utf-8", errors="ignore")

    # Parse the HTML document.
    soup = BeautifulSoup(html, "lxml")

    # Remove elements that are not part of the main documentation content.
    for tag in soup(["script", "style", "nav", "header", "footer", "aside"]):
        tag.decompose()

    # Try to select the main documentation content.
    main = (
        soup.find("main")
        or soup.find("div", class_=re.compile("body|document"))
        or soup.body
    )

    # Extract text from the main content area.
    text = main.get_text("\n", strip=True) if main else soup.get_text("\n", strip=True)

    # Clean repeated whitespace while preserving line breaks.
    lines = []

    for line in text.splitlines():
        line = re.sub(r"\s+", " ", line).strip()

        if not line:
            continue

        # Remove common documentation UI elements.
        if line in {
            "Navigation",
            "Theme",
            "Auto",
            "Light",
            "Dark",
            "previous",
            "next",
            "index",
            "modules",
        }:
            continue

        lines.append(line)

    # Return the cleaned document.
    return "\n".join(lines)


# Create a list for the processed documents.
documents = []


# Process every HTML file in the tutorial directory.
for path in sorted(TUTORIAL_DIR.glob("*.html")):
    # Clean the HTML content.
    text = html_to_text(path)

    # Store the text and its source metadata.
    documents.append({
        "source": path.name,
        "text": text,
        "length": len(text)
    })


# Convert the documents into a DataFrame.
df = pd.DataFrame(documents)


# Display dataset statistics.
print("Number of documents:", len(df))
print("Total characters:", df["length"].sum())
print("Average characters per document:", round(df["length"].mean()))

Number of documents: 17
Total characters: 234541
Average characters per document: 13797


In [11]:
# Select the first document for inspection.
sample = df.iloc[0]

# Display the source filename.
print("SOURCE:", sample["source"])
print("=" * 80)

# Display the first 3000 characters of the cleaned document.
print(sample["text"][:3000])

SOURCE: appendix.html
16.
Appendix
¶
16.1.
Interactive Mode
¶
16.1.1.
Error Handling
¶
When an error occurs, the interpreter prints an error message and a stack trace.
In interactive mode, it then returns to the primary prompt; when input came from
a file, it exits with a nonzero exit status after printing the stack trace.
(Exceptions handled by an
except
clause in a
try
statement
are not errors in this context.) Some errors are unconditionally fatal and
cause an exit with a nonzero exit; this applies to internal inconsistencies and
some cases of running out of memory. All error messages are written to the
standard error stream; normal output from executed commands is written to
standard output.
Typing the interrupt character (usually
Control
-
C
or
Delete
) to the primary or
secondary prompt cancels the input and returns to the primary prompt.
1
Typing an interrupt while a command is executing raises the
KeyboardInterrupt
exception, which may be handled by a
try
statement.
16.1.2.
Exe

In [12]:
# Define the chunk size and overlap used for splitting the documents.
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150


# Split a document into overlapping text chunks.
def create_chunks(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    # Store the generated chunks.
    chunks = []

    # Start from the beginning of the document.
    start = 0

    # Continue until the entire document has been processed.
    while start < len(text):
        # Calculate the end position of the current chunk.
        end = start + chunk_size

        # Extract the current chunk.
        chunk = text[start:end].strip()

        # Add non-empty chunks to the result.
        if chunk:
            chunks.append(chunk)

        # Move forward while keeping the specified overlap.
        start += chunk_size - overlap

    return chunks


# Create a list to store all chunks and their metadata.
chunked_documents = []


# Process every document in the dataset.
for _, row in df.iterrows():
    # Split the document into overlapping chunks.
    chunks = create_chunks(row["text"])

    # Store each chunk with its source and chunk index.
    for i, chunk in enumerate(chunks):
        chunked_documents.append({
            "source": row["source"],
            "chunk_id": i,
            "text": chunk
        })


# Convert the chunks into a DataFrame.
chunks_df = pd.DataFrame(chunked_documents)


# Display basic chunking statistics.
print("Total chunks:", len(chunks_df))
print("Average chunk length:",
      round(chunks_df["text"].str.len().mean()))

display(chunks_df.head())

Total chunks: 284
Average chunk length: 966


,source,chunk_id,text
0,appendix.html,0,16.\nAppendix\n¶\n16.1.\nInteractive Mode\n¶\n...
1,appendix.html,1,errupt while a command is executing raises the...
2,appendix.html,2,e Python\ninstaller automatically associates\n...
3,appendix.html,3,"ace where interactive commands are executed, s..."
4,appendix.html,4,"orks, you need first to find the location\nof ..."


In [13]:
# Inspect one generated chunk to verify the chunking strategy.
sample_chunk = chunks_df.iloc[0]

print("SOURCE:", sample_chunk["source"])
print("CHUNK ID:", sample_chunk["chunk_id"])
print("=" * 80)
print(sample_chunk["text"])

SOURCE: appendix.html
CHUNK ID: 0
16.
Appendix
¶
16.1.
Interactive Mode
¶
16.1.1.
Error Handling
¶
When an error occurs, the interpreter prints an error message and a stack trace.
In interactive mode, it then returns to the primary prompt; when input came from
a file, it exits with a nonzero exit status after printing the stack trace.
(Exceptions handled by an
except
clause in a
try
statement
are not errors in this context.) Some errors are unconditionally fatal and
cause an exit with a nonzero exit; this applies to internal inconsistencies and
some cases of running out of memory. All error messages are written to the
standard error stream; normal output from executed commands is written to
standard output.
Typing the interrupt character (usually
Control
-
C
or
Delete
) to the primary or
secondary prompt cancels the input and returns to the primary prompt.
1
Typing an interrupt while a command is executing raises the
KeyboardInterrupt
exception, which may be handled by a
try
statement.

In [14]:
# Validate the generated chunks before creating embeddings.
print("Total chunks:", len(chunks_df))

# Check for empty chunks.
empty_chunks = (chunks_df["text"].str.strip() == "").sum()

# Calculate the minimum and maximum chunk lengths.
min_length = chunks_df["text"].str.len().min()
max_length = chunks_df["text"].str.len().max()

print("Empty chunks:", empty_chunks)
print("Minimum chunk length:", min_length)
print("Maximum chunk length:", max_length)


# Display a few random chunks for manual inspection.
display(
    chunks_df.sample(
        min(3, len(chunks_df)),
        random_state=42
    )[["source", "chunk_id", "text"]]
)

Total chunks: 284
Empty chunks: 0
Minimum chunk length: 42
Maximum chunk length: 1000


,source,chunk_id,text
9,appetite.html,3,test functions\nduring bottom-up program devel...
253,stdlib.html,12,m Termination\n10.5. String Pattern Matching\n...
157,index.html,3,nd\nelse\nClauses on Loops\n4.5.\npass\nStatem...


In [15]:
# Install the libraries required for generating embeddings and storing vectors.
!pip install -q sentence-transformers chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 80.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 140.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 88.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 6.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currentl

In [16]:

# Import the SentenceTransformer model used to generate semantic embeddings.
from sentence_transformers import SentenceTransformer


# Load a lightweight pretrained embedding model.
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")


# Extract the text from all generated chunks.
chunk_texts = chunks_df["text"].tolist()


# Generate semantic vector embeddings for all chunks.
embeddings = embedding_model.encode(
    chunk_texts,
    show_progress_bar=True,
    normalize_embeddings=True
)


# Display the number and dimensionality of the generated embeddings.
print("Number of embeddings:", len(embeddings))
print("Embedding dimension:", embeddings.shape[1])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/9 [00:00<?, ?it/s]

Number of embeddings: 284
Embedding dimension: 384


In [17]:
# Import ChromaDB for storing and retrieving document embeddings.
import chromadb


# Define the directory where the vector database will be persisted.
CHROMA_PATH = "/content/chroma_db"


# Create a persistent ChromaDB client.
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)


# Create or load the collection used by the Python Learning Assistant.
collection = chroma_client.get_or_create_collection(
    name="python_tutorial"
)


# Display information about the collection.
print("Collection name:", collection.name)
print("Existing documents:", collection.count())

Collection name: python_tutorial
Existing documents: 0


In [18]:
# Prepare unique IDs for all document chunks.
chunk_ids = [
    f"{row['source']}_{row['chunk_id']}"
    for _, row in chunks_df.iterrows()
]


# Prepare metadata for each chunk.
metadatas = [
    {
        "source": row["source"],
        "chunk_id": int(row["chunk_id"])
    }
    for _, row in chunks_df.iterrows()
]


# Store the chunks, embeddings, and metadata in ChromaDB.
collection.add(
    ids=chunk_ids,
    documents=chunk_texts,
    embeddings=embeddings.tolist(),
    metadatas=metadatas
)


# Verify that all chunks were stored successfully.
print("Documents stored in ChromaDB:", collection.count())

Documents stored in ChromaDB: 284


In [19]:
# Create a function that retrieves the most relevant document chunks for a user question.
def retrieve_documents(question, top_k=5):
    # Generate an embedding for the user's question.
    query_embedding = embedding_model.encode(
        [question],
        normalize_embeddings=True
    )[0]

    # Search the ChromaDB collection using the question embedding.
    results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=top_k
    )

    # Return the retrieved results.
    return results

In [20]:
# Define a sample question about Python.
question = "What is a Python list?"


# Retrieve the five most relevant chunks from the vector database.
results = retrieve_documents(question, top_k=5)


# Display the retrieved chunks and their sources.
for i, (document, metadata, distance) in enumerate(
    zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ),
    start=1
):
    print(f"Result {i}")
    print("Source:", metadata["source"])
    print("Chunk ID:", metadata["chunk_id"])
    print("Distance:", round(distance, 4))
    print("Text:")
    print(document[:700])
    print("=" * 80)

Result 1
Source: controlflow.html
Chunk ID: 18
Distance: 0.8308
Text:
2, 3, 5, 8, 13, 21, 34, 55, 89]
This example, as usual, demonstrates some new Python features:
The
return
statement returns with a value from a function.
return
without an expression argument returns
None
. Falling off
the end of a function also returns
None
.
The statement
result.append(a)
calls a
method
of the list object
result
. A method is a function that ‘belongs’ to an object and is named
obj.methodname
, where
obj
is some object (this may be an expression),
and
methodname
is the name of a method that is defined by the object’s type.
Different types define different methods. Methods of different types may have
the same name without causing ambiguity. (It is possible to define your own
Result 2
Source: datastructures.html
Chunk ID: 0
Distance: 0.8323
Text:
5.
Data Structures
¶
This chapter describes some things you’ve learned about already in more detail,
and adds some new things as well.
5.1.
More on Lists
¶
T

In [21]:
# Define representative questions from different Python Tutorial topics.
test_questions = [
    "What is a Python list?",
    "How does a for loop work in Python?",
    "What is a Python dictionary?",
    "How do you handle exceptions in Python?",
    "What is a Python class?"
]


# Retrieve the top 3 relevant chunks for each question.
for question in test_questions:
    print("\nQUESTION:", question)
    print("-" * 80)

    results = retrieve_documents(question, top_k=3)

    for i, metadata in enumerate(results["metadatas"][0], start=1):
        distance = results["distances"][0][i - 1]

        print(
            f"{i}. Source: {metadata['source']} | "
            f"Chunk: {metadata['chunk_id']} | "
            f"Distance: {distance:.4f}"
        )


QUESTION: What is a Python list?
--------------------------------------------------------------------------------
1. Source: controlflow.html | Chunk: 18 | Distance: 0.8308
2. Source: datastructures.html | Chunk: 0 | Distance: 0.8323
3. Source: datastructures.html | Chunk: 11 | Distance: 0.8519

QUESTION: How does a for loop work in Python?
--------------------------------------------------------------------------------
1. Source: controlflow.html | Chunk: 3 | Distance: 0.8184
2. Source: classes.html | Chunk: 35 | Distance: 0.8350
3. Source: controlflow.html | Chunk: 1 | Distance: 0.8699

QUESTION: What is a Python dictionary?
--------------------------------------------------------------------------------
1. Source: classes.html | Chunk: 39 | Distance: 0.8552
2. Source: appetite.html | Chunk: 2 | Distance: 0.9049
3. Source: controlflow.html | Chunk: 33 | Distance: 0.9721

QUESTION: How do you handle exceptions in Python?
---------------------------------------------------------------

In [22]:
# Retrieve and display the full context for one representative question.
question = "How do you handle exceptions in Python?"

results = retrieve_documents(question, top_k=3)


# Combine the retrieved chunks into a single context string.
retrieved_context = "\n\n".join(
    results["documents"][0]
)


# Display the question and retrieved context.
print("QUESTION:")
print(question)

print("\n" + "=" * 80)
print("RETRIEVED CONTEXT")
print("=" * 80)

print(retrieved_context)

QUESTION:
How do you handle exceptions in Python?

RETRIEVED CONTEXT
caution, since it is easy to mask a real
programming error in this way! It can also be used to print an error message and
then re-raise the exception (allowing a caller to handle the exception as well):
import
sys
try
:
f
=
open
(
'myfile.txt'
)
s
=
f
.
readline
()
i
=
int
(
s
.
strip
())
except
OSError
as
err
:
print
(
"OS error:
{0}
"
.
format
(
err
))
except
ValueError
:
print
(
"Could not convert data to an integer."
)
except
BaseException
as
err
:
print
(
f
"Unexpected
{
err
=}
,
{
type
(
err
)
=}
"
)
raise
Alternatively the last except clause may omit the exception name(s), however the exception
value must then be retrieved from
sys.exc_info()[1]
.
The
try
…
except
statement has an optional
else
clause
, which, when present, must follow all
except clauses
. It is useful
for code that must be executed if the
try clause
does not raise an exception.
For example:
for
arg
in
sys
.
argv
[
1
:]:
try
:
f
=
open
(
arg
,


In [23]:
# Test the retrieved context for the Python dictionary question.
question = "What is a Python dictionary?"


# Retrieve the top 5 relevant chunks.
results = retrieve_documents(question, top_k=5)


# Display each retrieved chunk with its source and distance.
for i, (document, metadata, distance) in enumerate(
    zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ),
    start=1
):
    print(f"RESULT {i}")
    print("Source:", metadata["source"])
    print("Chunk ID:", metadata["chunk_id"])
    print("Distance:", round(distance, 4))
    print("Text:")
    print(document)
    print("\n" + "=" * 100 + "\n")


RESULT 1
Source: classes.html
Chunk ID: 39
Distance: 0.8552
Text:
unique_words
=
set
(
word
for
line
in
page
for
word
in
line
.
split
())
>>>
valedictorian
=
max
((
student
.
gpa
,
student
.
name
)
for
student
in
graduates
)
>>>
data
=
'golf'
>>>
list
(
data
[
i
]
for
i
in
range
(
len
(
data
)
-
1
,
-
1
,
-
1
))
['f', 'l', 'o', 'g']
Footnotes
1
Except for one thing. Module objects have a secret read-only attribute called
__dict__
which returns the dictionary used to implement the module’s
namespace; the name
__dict__
is an attribute but not a global name.
Obviously, using this violates the abstraction of namespace implementation, and
should be restricted to things like post-mortem debuggers.
Table of Contents
9. Classes
9.1. A Word About Names and Objects
9.2. Python Scopes and Namespaces
9.2.1. Scopes and Namespaces Example
9.3. A First Look at Classes
9.3.1. Class Definition Syntax
9.3.2. Class Objects
9.3.3. Instance Objects
9.3.4. Method Objects
9.3.5. Class and Instance Variables


In [24]:
# Import libraries required for improved HTML document extraction.
from pathlib import Path
from bs4 import BeautifulSoup
import re
import pandas as pd


# Define the directory containing the Python Tutorial HTML files.
TUTORIAL_DIR = Path("/content/tutorial/tutorial")


# Extract clean, structured text from a Python documentation HTML file.
def extract_document(path):
    # Read the original HTML file.
    html = path.read_text(encoding="utf-8", errors="ignore")

    # Parse the HTML document.
    soup = BeautifulSoup(html, "lxml")

    # Remove website elements that are not part of the documentation.
    for tag in soup(["script", "style", "nav", "header", "footer", "aside"]):
        tag.decompose()

    # Find the main documentation container.
    main = soup.find("main")

    if main is None:
        main = soup.find("div", class_=re.compile("body|document"))

    if main is None:
        main = soup.body

    # Store extracted sections.
    sections = []

    # Extract headings, paragraphs, lists, and code blocks in document order.
    for element in main.find_all(
        ["h1", "h2", "h3", "h4", "h5", "p", "li", "pre"]
    ):
        text = element.get_text(" ", strip=True)

        # Normalize whitespace.
        text = re.sub(r"\s+", " ", text).strip()

        # Ignore empty elements.
        if not text:
            continue

        # Preserve headings with a clear structure.
        if element.name in ["h1", "h2", "h3", "h4", "h5"]:
            sections.append(f"\n{text}\n")

        # Preserve code examples as separate blocks.
        elif element.name == "pre":
            sections.append(f"\nCODE:\n{text}\n")

        # Preserve normal documentation text.
        else:
            sections.append(text)

    # Join the extracted sections.
    document_text = "\n".join(sections)

    # Remove excessive blank lines.
    document_text = re.sub(r"\n{3,}", "\n\n", document_text)

    return document_text.strip()


# Extract all Python Tutorial documents.
documents = []

for path in sorted(TUTORIAL_DIR.glob("*.html")):
    text = extract_document(path)

    documents.append({
        "source": path.name,
        "text": text,
        "length": len(text)
    })


# Convert the processed documents into a DataFrame.
df_improved = pd.DataFrame(documents)


# Display basic statistics.
print("Documents:", len(df_improved))
print("Total characters:", df_improved["length"].sum())
print("Average document length:", round(df_improved["length"].mean()))

Documents: 17
Total characters: 261468
Average document length: 15380


In [25]:
# Inspect the improved extraction from the data structures documentation.
sample = df_improved[
    df_improved["source"] == "datastructures.html"
].iloc[0]


# Display the first 4000 characters.
print("SOURCE:", sample["source"])
print("=" * 80)
print(sample["text"][:4000])

SOURCE: datastructures.html
5. Data Structures ¶

This chapter describes some things you’ve learned about already in more detail, and adds some new things as well.

5.1. More on Lists ¶

The list data type has some more methods. Here are all of the methods of list objects:
Add an item to the end of the list. Equivalent to a[len(a):] = [x] .
Extend the list by appending all the items from the iterable. Equivalent to a[len(a):] = iterable .
Insert an item at a given position. The first argument is the index of the element before which to insert, so a.insert(0, x) inserts at the front of the list, and a.insert(len(a), x) is equivalent to a.append(x) .
Remove the first item from the list whose value is equal to x . It raises a ValueError if there is no such item.
Remove the item at the given position in the list, and return it. If no index is specified, a.pop() removes and returns the last item in the list. (The square brackets around the i in the method signature denote that the parameter

In [26]:
# Clean extracted text while preserving headings, paragraphs, lists, and code blocks.
def clean_extracted_text(text):
    # Remove heading markers that come from the HTML documentation.
    text = text.replace(" ¶", "")

    # Normalize excessive spaces while keeping line breaks.
    lines = []
    for line in text.splitlines():
        line = re.sub(r"[ \t]+", " ", line).strip()

        # Skip empty lines.
        if not line:
            continue

        lines.append(line)

    # Remove excessive blank lines.
    text = "\n".join(lines)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


# Apply the cleaning step to every extracted document.
df_improved["text"] = df_improved["text"].apply(clean_extracted_text)

# Recalculate document lengths after cleaning.
df_improved["length"] = df_improved["text"].str.len()

# Display basic statistics for validation.
print("Documents:", len(df_improved))
print("Total characters:", df_improved["length"].sum())
print("Average document length:", round(df_improved["length"].mean()))

# Check the beginning of the data structures document again.
sample = df_improved[
    df_improved["source"] == "datastructures.html"
].iloc[0]

print("\nSOURCE:", sample["source"])
print("=" * 80)
print(sample["text"][:2500])

Documents: 17
Total characters: 260215
Average document length: 15307

SOURCE: datastructures.html
5. Data Structures
This chapter describes some things you’ve learned about already in more detail, and adds some new things as well.
5.1. More on Lists
The list data type has some more methods. Here are all of the methods of list objects:
Add an item to the end of the list. Equivalent to a[len(a):] = [x] .
Extend the list by appending all the items from the iterable. Equivalent to a[len(a):] = iterable .
Insert an item at a given position. The first argument is the index of the element before which to insert, so a.insert(0, x) inserts at the front of the list, and a.insert(len(a), x) is equivalent to a.append(x) .
Remove the first item from the list whose value is equal to x . It raises a ValueError if there is no such item.
Remove the item at the given position in the list, and return it. If no index is specified, a.pop() removes and returns the last item in the list. (The square bracket

In [27]:
# Create section-aware chunks from the cleaned documentation.
# Each chunk keeps the most recent section heading as part of its context.

CHUNK_SIZE = 1200
CHUNK_OVERLAP = 150

def create_section_chunks(text, source, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    # Split the document into non-empty lines.
    lines = [line.strip() for line in text.splitlines() if line.strip()]

    chunks = []
    current_heading = ""
    current_text = []

    for line in lines:
        # Detect documentation headings using the Python docs numbering style.
        if re.match(r"^\d+(\.\d+)*\.\s+", line):
            # Save the previous section before starting a new one.
            if current_text:
                section_text = "\n".join(current_text).strip()

                # Split long sections into overlapping chunks.
                start = 0
                while start < len(section_text):
                    end = start + chunk_size
                    chunk_body = section_text[start:end].strip()

                    if chunk_body:
                        chunk_text = (
                            f"Section: {current_heading}\n"
                            f"{chunk_body}"
                        )

                        chunks.append({
                            "source": source,
                            "section": current_heading,
                            "text": chunk_text
                        })

                    # Move forward while keeping overlap between chunks.
                    start += chunk_size - overlap

            # Start the new section.
            current_heading = line
            current_text = []

        else:
            # Add normal content to the current section.
            current_text.append(line)

    # Process the final section in the document.
    if current_text:
        section_text = "\n".join(current_text).strip()

        start = 0
        while start < len(section_text):
            end = start + chunk_size
            chunk_body = section_text[start:end].strip()

            if chunk_body:
                chunk_text = (
                    f"Section: {current_heading}\n"
                    f"{chunk_body}"
                )

                chunks.append({
                    "source": source,
                    "section": current_heading,
                    "text": chunk_text
                })

            start += chunk_size - overlap

    return chunks


# Create the new chunk collection from all cleaned documents.
section_chunks = []

for _, row in df_improved.iterrows():
    document_chunks = create_section_chunks(
        row["text"],
        row["source"]
    )

    # Add a sequential chunk ID for each source document.
    for chunk_id, chunk in enumerate(document_chunks):
        chunk["chunk_id"] = chunk_id
        section_chunks.append(chunk)


# Convert the chunks into a DataFrame for inspection and later embedding.
section_chunks_df = pd.DataFrame(section_chunks)

# Display chunk statistics.
print("Total chunks:", len(section_chunks_df))
print("Average chunk length:",
      round(section_chunks_df["text"].str.len().mean()))
print("Minimum chunk length:",
      section_chunks_df["text"].str.len().min())
print("Maximum chunk length:",
      section_chunks_df["text"].str.len().max())

Total chunks: 332
Average chunk length: 824
Minimum chunk length: 30
Maximum chunk length: 1270


In [29]:
# Inspect several chunks from the data structures documentation.
sample_chunks = section_chunks_df[
    section_chunks_df["source"] == "datastructures.html"
].head(5)

for _, row in sample_chunks.iterrows():
    print("=" * 80)
    print("SOURCE:", row["source"])
    print("SECTION:", row["section"])
    print("CHUNK ID:", row["chunk_id"])
    print(row["text"][:1200])
    print()

SOURCE: datastructures.html
SECTION: 5. Data Structures
CHUNK ID: 0
Section: 5. Data Structures
This chapter describes some things you’ve learned about already in more detail, and adds some new things as well.

SOURCE: datastructures.html
SECTION: 5.1. More on Lists
CHUNK ID: 1
Section: 5.1. More on Lists
The list data type has some more methods. Here are all of the methods of list objects:
Add an item to the end of the list. Equivalent to a[len(a):] = [x] .
Extend the list by appending all the items from the iterable. Equivalent to a[len(a):] = iterable .
Insert an item at a given position. The first argument is the index of the element before which to insert, so a.insert(0, x) inserts at the front of the list, and a.insert(len(a), x) is equivalent to a.append(x) .
Remove the first item from the list whose value is equal to x . It raises a ValueError if there is no such item.
Remove the item at the given position in the list, and return it. If no index is specified, a.pop() removes an

In [30]:
# Generate semantic embeddings for the new section-aware chunks.
# The same embedding model is used for both documents and user questions.

from sentence_transformers import SentenceTransformer

# Load the embedding model used for semantic retrieval.
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Extract the chunk texts that will be embedded.
section_chunk_texts = section_chunks_df["text"].tolist()

# Generate normalized embeddings for all chunks.
section_embeddings = embedding_model.encode(
    section_chunk_texts,
    show_progress_bar=True,
    normalize_embeddings=True
)

# Display the embedding statistics.
print("Number of embeddings:", len(section_embeddings))
print("Embedding dimension:", section_embeddings.shape[1])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

Number of embeddings: 332
Embedding dimension: 384


In [31]:
# Create a separate persistent ChromaDB collection for the improved section-aware chunks.
# The old collection is kept unchanged as a backup for comparison.

import chromadb

# Define a separate directory for the improved vector store.
CHROMA_PATH_V2 = "/content/chroma_db_v2"

# Create a persistent ChromaDB client.
chroma_client_v2 = chromadb.PersistentClient(path=CHROMA_PATH_V2)

# Create a new collection for the improved chunks.
collection_v2 = chroma_client_v2.get_or_create_collection(
    name="python_tutorial_v2"
)

# Display the current number of stored chunks.
print("Collection name:", collection_v2.name)
print("Existing documents:", collection_v2.count())

Collection name: python_tutorial_v2
Existing documents: 0


In [32]:
# Store the improved chunks, embeddings, and metadata in ChromaDB.

# Create unique IDs for every chunk.
chunk_ids_v2 = [
    f"{row['source']}_{row['chunk_id']}"
    for _, row in section_chunks_df.iterrows()
]

# Store metadata needed for source citation and traceability.
metadatas_v2 = [
    {
        "source": row["source"],
        "section": row["section"],
        "chunk_id": int(row["chunk_id"])
    }
    for _, row in section_chunks_df.iterrows()
]

# Add the documents, embeddings, and metadata to the new collection.
collection_v2.add(
    ids=chunk_ids_v2,
    documents=section_chunk_texts,
    embeddings=section_embeddings.tolist(),
    metadatas=metadatas_v2
)

# Verify that all chunks were stored successfully.
print("Documents stored in ChromaDB:", collection_v2.count())

Documents stored in ChromaDB: 332


In [33]:
# Define the retrieval function for the improved section-aware vector store.
# The same embedding model is used to embed the user's question.

def retrieve_documents_v2(question, top_k=5):
    # Convert the question into a normalized semantic embedding.
    query_embedding = embedding_model.encode(
        [question],
        normalize_embeddings=True
    )[0]

    # Search the persistent Chroma collection for the most similar chunks.
    results = collection_v2.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=top_k
    )

    return results

In [34]:
# Test the improved retrieval system using the dictionary question.
# This question previously produced weak retrieval results.

question = "What is a Python dictionary?"

results = retrieve_documents_v2(question, top_k=5)

# Display the retrieved chunks and their metadata.
for i in range(5):
    print("=" * 80)
    print("Rank:", i + 1)
    print("Source:", results["metadatas"][0][i]["source"])
    print("Section:", results["metadatas"][0][i]["section"])
    print("Chunk ID:", results["metadatas"][0][i]["chunk_id"])
    print("Distance:", round(results["distances"][0][i], 4))
    print("Text:")
    print(results["documents"][0][i][:1000])
    print()

Rank: 1
Source: datastructures.html
Section: 5.5. Dictionaries
Chunk ID: 17
Distance: 0.7852
Text:
Section: 5.5. Dictionaries
Another useful data type built into Python is the dictionary (see Mapping Types — dict ). Dictionaries are sometimes found in other languages as “associative memories” or “associative arrays”. Unlike sequences, which are indexed by a range of numbers, dictionaries are indexed by keys , which can be any immutable type; strings and numbers can always be keys. Tuples can be used as keys if they contain only strings, numbers, or tuples; if a tuple contains any mutable object either directly or indirectly, it cannot be used as a key. You can’t use lists as keys, since lists can be modified in place using index assignments, slice assignments, or methods like append() and extend() .
It is best to think of a dictionary as a set of key: value pairs, with the requirement that the keys are unique (within one dictionary). A pair of braces creates an empty dictionary: {} . P

In [35]:
# Evaluate the improved retrieval system on several representative Python questions.
# The questions cover different topics from the official Python tutorial.

test_questions = [
    "What is a Python list?",
    "How does a for loop work in Python?",
    "What is a Python dictionary?",
    "How do you handle exceptions in Python?",
    "What is a Python class?"
]

# Retrieve the top 5 chunks for each test question.
for question in test_questions:
    results = retrieve_documents_v2(question, top_k=5)

    print("=" * 100)
    print("QUESTION:", question)

    # Display the metadata of the five retrieved chunks.
    for rank in range(5):
        metadata = results["metadatas"][0][rank]
        distance = results["distances"][0][rank]

        print(
            f"{rank + 1}. "
            f"{metadata['source']} | "
            f"{metadata['section']} | "
            f"chunk={metadata['chunk_id']} | "
            f"distance={distance:.4f}"
        )

    print()

QUESTION: What is a Python list?
1. datastructures.html | 5.1. More on Lists | chunk=1 | distance=0.7415
2. datastructures.html | 5.5. Dictionaries | chunk=17 | distance=0.7895
3. introduction.html | 3.1.3. Lists | chunk=13 | distance=0.7951
4. interpreter.html | 3. An Informal Introduction to Python | chunk=9 | distance=0.8566
5. index.html |  | chunk=0 | distance=0.8604

QUESTION: How does a for loop work in Python?
1. classes.html | 9.8. Iterators | chunk=34 | distance=0.7202
2. controlflow.html | 4.2. for Statements | chunk=2 | distance=0.8067
3. datastructures.html | 5.1.3. List Comprehensions | chunk=6 | distance=0.8893
4. introduction.html | 3.2. First Steps Towards Programming | chunk=18 | distance=0.9086
5. datastructures.html | 5.6. Looping Techniques | chunk=20 | distance=0.9108

QUESTION: What is a Python dictionary?
1. datastructures.html | 5.5. Dictionaries | chunk=17 | distance=0.7852
2. whatnow.html | 13. What Now? | chunk=1 | distance=0.8820
3. index.html |  | chunk=0 

In [36]:
# Build a clean context string from the retrieved documents.
# The context will be passed to the LLM as the only source of information.

def build_context(results):
    # Store the formatted retrieved chunks.
    context_parts = []

    # Iterate through every retrieved document.
    for i in range(len(results["documents"][0])):
        document = results["documents"][0][i]
        metadata = results["metadatas"][0][i]

        # Extract source metadata for citation grounding.
        source = metadata["source"]
        section = metadata["section"]

        # Format each retrieved chunk with its source information.
        context_parts.append(
            f"[Source: {source} | Section: {section}]\n"
            f"{document}"
        )

    # Separate retrieved chunks clearly.
    return "\n\n---\n\n".join(context_parts)

In [37]:
# Test the context-building function with the dictionary question.

question = "What is a Python dictionary?"

# Retrieve the most relevant documents.
results = retrieve_documents_v2(question, top_k=5)

# Build the context that will later be sent to the LLM.
context = build_context(results)

# Display the generated context.
print(context)

[Source: datastructures.html | Section: 5.5. Dictionaries]
Section: 5.5. Dictionaries
Another useful data type built into Python is the dictionary (see Mapping Types — dict ). Dictionaries are sometimes found in other languages as “associative memories” or “associative arrays”. Unlike sequences, which are indexed by a range of numbers, dictionaries are indexed by keys , which can be any immutable type; strings and numbers can always be keys. Tuples can be used as keys if they contain only strings, numbers, or tuples; if a tuple contains any mutable object either directly or indirectly, it cannot be used as a key. You can’t use lists as keys, since lists can be modified in place using index assignments, slice assignments, or methods like append() and extend() .
It is best to think of a dictionary as a set of key: value pairs, with the requirement that the keys are unique (within one dictionary). A pair of braces creates an empty dictionary: {} . Placing a comma-separated list of key:val

In [38]:
# Create the prompt used to generate grounded answers from retrieved documentation.
# The model must answer only from the provided context and cite its sources.

def build_rag_prompt(question, context):
    # Define strict instructions to prevent unsupported answers.
    system_instruction = """
You are a Python learning assistant.

Answer the user's question using ONLY the provided context from the official
Python documentation.

Rules:
1. Do not use outside knowledge.
2. If the context does not contain enough information, say:
   "I couldn't find enough information in the provided Python documentation."
3. Give a clear and concise explanation suitable for a Python learner.
4. Cite the relevant source filename in the answer using [source: filename].
5. Do not invent sources or citations.
""".strip()

    # Combine the instructions, retrieved context, and user question.
    prompt = f"""
{system_instruction}

CONTEXT:
{context}

QUESTION:
{question}

ANSWER:
""".strip()

    return prompt

In [39]:
# Build and inspect the final RAG prompt for the dictionary question.

question = "What is a Python dictionary?"

# Retrieve the relevant documentation chunks.
results = retrieve_documents_v2(question, top_k=5)

# Build the grounded context.
context = build_context(results)

# Build the final prompt that will be sent to the LLM.
rag_prompt = build_rag_prompt(question, context)

# Display the prompt for inspection.
print(rag_prompt)

You are a Python learning assistant.

Answer the user's question using ONLY the provided context from the official
Python documentation.

Rules:
1. Do not use outside knowledge.
2. If the context does not contain enough information, say:
   "I couldn't find enough information in the provided Python documentation."
3. Give a clear and concise explanation suitable for a Python learner.
4. Cite the relevant source filename in the answer using [source: filename].
5. Do not invent sources or citations.

CONTEXT:
[Source: datastructures.html | Section: 5.5. Dictionaries]
Section: 5.5. Dictionaries
Another useful data type built into Python is the dictionary (see Mapping Types — dict ). Dictionaries are sometimes found in other languages as “associative memories” or “associative arrays”. Unlike sequences, which are indexed by a range of numbers, dictionaries are indexed by keys , which can be any immutable type; strings and numbers can always be keys. Tuples can be used as keys if they contai

In [40]:
# Create a strict grounded RAG prompt for the language model.
# The model is instructed to rely only on retrieved documentation.

def build_rag_prompt(question, context):
    # Define the system instructions for grounded generation.
    system_instruction = """
You are a Python learning assistant.

Answer the user's question using ONLY the provided context from the official
Python documentation.

Rules:
1. Do not use outside knowledge.
2. If the context does not contain enough information, say:
   "I couldn't find enough information in the provided Python documentation."
3. Give a clear and concise explanation suitable for a Python learner.
4. Cite supporting sources using the exact format [source: filename].
5. Only cite filenames that actually appear in the provided context.
6. Do not invent facts, sources, or citations.
""".strip()

    # Combine the instructions, retrieved context, and user question.
    prompt = f"""
{system_instruction}

CONTEXT:
{context}

QUESTION:
{question}

ANSWER:
""".strip()

    return prompt

In [41]:
# Test the grounded prompt using the three most relevant retrieved chunks.
# Using fewer high-quality chunks helps reduce irrelevant context.

question = "What is a Python dictionary?"

# Retrieve the three most relevant chunks.
results = retrieve_documents_v2(question, top_k=3)

# Build the context from the retrieved chunks.
context = build_context(results)

# Build the final grounded prompt.
rag_prompt = build_rag_prompt(question, context)

# Display the final prompt.
print(rag_prompt)

You are a Python learning assistant.

Answer the user's question using ONLY the provided context from the official
Python documentation.

Rules:
1. Do not use outside knowledge.
2. If the context does not contain enough information, say:
   "I couldn't find enough information in the provided Python documentation."
3. Give a clear and concise explanation suitable for a Python learner.
4. Cite supporting sources using the exact format [source: filename].
5. Only cite filenames that actually appear in the provided context.
6. Do not invent facts, sources, or citations.

CONTEXT:
[Source: datastructures.html | Section: 5.5. Dictionaries]
Section: 5.5. Dictionaries
Another useful data type built into Python is the dictionary (see Mapping Types — dict ). Dictionaries are sometimes found in other languages as “associative memories” or “associative arrays”. Unlike sequences, which are indexed by a range of numbers, dictionaries are indexed by keys , which can be any immutable type; strings and

In [42]:
# Store the main RAG configuration in one place.
# This configuration will later be reused by the backend.

RAG_CONFIG = {
    "embedding_model": "all-MiniLM-L6-v2",
    "vector_store": "ChromaDB",
    "collection_name": "python_tutorial_v2",
    "chunking_strategy": "section-aware chunks with 1200-character maximum and 150-character overlap",
    "top_k": 3,
    "domain": "Python Programming",
    "source": "Official Python 3.10 Tutorial"
}

# Display the configuration for verification.
for key, value in RAG_CONFIG.items():
    print(f"{key}: {value}")

embedding_model: all-MiniLM-L6-v2
vector_store: ChromaDB
collection_name: python_tutorial_v2
chunking_strategy: section-aware chunks with 1200-character maximum and 150-character overlap
top_k: 3
domain: Python Programming
source: Official Python 3.10 Tutorial


In [43]:
# Create a structured retrieval evaluation for the initial five questions.
# The expected sections are based on the official Python tutorial structure.

evaluation_questions = [
    {
        "question": "What is a Python list?",
        "expected_section": "5.1. More on Lists"
    },
    {
        "question": "How does a for loop work in Python?",
        "expected_section": "4.2. for Statements"
    },
    {
        "question": "What is a Python dictionary?",
        "expected_section": "5.5. Dictionaries"
    },
    {
        "question": "How do you handle exceptions in Python?",
        "expected_section": "8.3. Handling Exceptions"
    },
    {
        "question": "What is a Python class?",
        "expected_section": "9. Classes"
    }
]

# Store the evaluation results.
evaluation_results = []

# Evaluate whether the expected section appears within the top 5 results.
for item in evaluation_questions:
    results = retrieve_documents_v2(item["question"], top_k=5)

    retrieved_sections = [
        metadata["section"]
        for metadata in results["metadatas"][0]
    ]

    top_1_section = retrieved_sections[0]

    # Check whether the expected section was retrieved in the top 5.
    retrieved_correctly = item["expected_section"] in retrieved_sections

    evaluation_results.append({
        "question": item["question"],
        "expected_section": item["expected_section"],
        "top_1_section": top_1_section,
        "top_5_sections": " | ".join(retrieved_sections),
        "retrieved_correctly": retrieved_correctly
    })

# Convert the evaluation results into a DataFrame.
evaluation_df = pd.DataFrame(evaluation_results)

# Display the evaluation table.
evaluation_df

,question,expected_section,top_1_section,top_5_sections,retrieved_correctly
0,What is a Python list?,5.1. More on Lists,5.1. More on Lists,5.1. More on Lists | 5.5. Dictionaries | 3.1.3...,True
1,How does a for loop work in Python?,4.2. for Statements,9.8. Iterators,9.8. Iterators | 4.2. for Statements | 5.1.3. ...,True
2,What is a Python dictionary?,5.5. Dictionaries,5.5. Dictionaries,5.5. Dictionaries | 13. What Now? | | 1. Whet...,True
3,How do you handle exceptions in Python?,8.3. Handling Exceptions,8.3. Handling Exceptions,8.3. Handling Exceptions | 8.3. Handling Excep...,True
4,What is a Python class?,9. Classes,9. Classes,9. Classes | 9.3.2. Class Objects | 9.2. Pytho...,True


In [44]:
# Define 10 representative questions for evaluating retrieval quality.
evaluation_questions = [
    {
        "question": "What is a Python list?",
        "expected_section": "5.1. More on Lists"
    },
    {
        "question": "How does a for loop work in Python?",
        "expected_section": "4.2. for Statements"
    },
    {
        "question": "What is a Python dictionary?",
        "expected_section": "5.5. Dictionaries"
    },
    {
        "question": "How do you handle exceptions in Python?",
        "expected_section": "8.3. Handling Exceptions"
    },
    {
        "question": "What is a Python class?",
        "expected_section": "9. Classes"
    },
    {
        "question": "How do you define a function in Python?",
        "expected_section": "4.8. Defining Functions"
    },
    {
        "question": "How can you read and write files in Python?",
        "expected_section": "7.2. Reading and Writing Files"
    },
    {
        "question": "What are list comprehensions in Python?",
        "expected_section": "5.1.3. List Comprehensions"
    },
    {
        "question": "What is a virtual environment in Python?",
        "expected_section": "12.2. Creating Virtual Environments"
    },
    {
        "question": "How does Python import modules?",
        "expected_section": "6.1. More on Modules"
    }
]

print(f"Number of evaluation questions: {len(evaluation_questions)}")

Number of evaluation questions: 10


In [45]:
# Evaluate retrieval performance across all 10 questions.
evaluation_results = []

for item in evaluation_questions:
    results = retrieve_documents_v2(item["question"], top_k=5)

    retrieved_sections = [
        metadata["section"]
        for metadata in results["metadatas"][0]
    ]

    top_1_section = retrieved_sections[0]

    retrieved_correctly = (
        item["expected_section"] in retrieved_sections
    )

    evaluation_results.append({
        "question": item["question"],
        "expected_section": item["expected_section"],
        "top_1_section": top_1_section,
        "top_5_sections": " | ".join(retrieved_sections),
        "retrieved_correctly": retrieved_correctly
    })

evaluation_df = pd.DataFrame(evaluation_results)

evaluation_df

,question,expected_section,top_1_section,top_5_sections,retrieved_correctly
0,What is a Python list?,5.1. More on Lists,5.1. More on Lists,5.1. More on Lists | 5.5. Dictionaries | 3.1.3...,True
1,How does a for loop work in Python?,4.2. for Statements,9.8. Iterators,9.8. Iterators | 4.2. for Statements | 5.1.3. ...,True
2,What is a Python dictionary?,5.5. Dictionaries,5.5. Dictionaries,5.5. Dictionaries | 13. What Now? | | 1. Whet...,True
3,How do you handle exceptions in Python?,8.3. Handling Exceptions,8.3. Handling Exceptions,8.3. Handling Exceptions | 8.3. Handling Excep...,True
4,What is a Python class?,9. Classes,9. Classes,9. Classes | 9.3.2. Class Objects | 9.2. Pytho...,True
5,How do you define a function in Python?,4.8. Defining Functions,4.8.3. Special parameters,4.8.3. Special parameters | 4.7. Defining Func...,False
6,How can you read and write files in Python?,7.2. Reading and Writing Files,7.2.1. Methods of File Objects,7.2.1. Methods of File Objects | 7.2. Reading ...,True
7,What are list comprehensions in Python?,5.1.3. List Comprehensions,5.1.3. List Comprehensions,5.1.3. List Comprehensions | 5.5. Dictionaries...,True
8,What is a virtual environment in Python?,12.2. Creating Virtual Environments,12.2. Creating Virtual Environments,12.2. Creating Virtual Environments | 12.2. Cr...,True
9,How does Python import modules?,6.1. More on Modules,6.1.2. The Module Search Path,6.1.2. The Module Search Path | 6.1. More on M...,True


In [46]:
# Calculate retrieval accuracy based on whether the expected section appears in the top-5 results.
retrieval_accuracy = evaluation_df["retrieved_correctly"].mean()

print(f"Retrieval Accuracy (Top-5): {retrieval_accuracy:.2%}")

Retrieval Accuracy (Top-5): 90.00%


In [47]:
# Display only the evaluation questions where the expected section was not retrieved.
failed_retrievals = evaluation_df[
    evaluation_df["retrieved_correctly"] == False
]

failed_retrievals

,question,expected_section,top_1_section,top_5_sections,retrieved_correctly
5,How do you define a function in Python?,4.8. Defining Functions,4.8.3. Special parameters,4.8.3. Special parameters | 4.7. Defining Func...,False


In [48]:
# Inspect the exact retrieved section strings for the function-definition question.
question = "How do you define a function in Python?"

results = retrieve_documents_v2(question, top_k=5)

for rank, metadata in enumerate(results["metadatas"][0], start=1):
    print(f"Rank {rank}:")
    print(f"Section: {repr(metadata['section'])}")
    print(f"Source: {metadata['source']}")
    print()

Rank 1:
Section: '4.8.3. Special parameters'
Source: controlflow.html

Rank 2:
Section: '4.7. Defining Functions'
Source: controlflow.html

Rank 3:
Section: '9.3.4. Method Objects'
Source: classes.html

Rank 4:
Section: '4.7. Defining Functions'
Source: controlflow.html

Rank 5:
Section: '4.8.1. Default Argument Values'
Source: controlflow.html



In [49]:
# Compare the expected section with the retrieved sections using exact string matching.
expected_section = "4.8. Defining Functions"

retrieved_sections = [
    metadata["section"]
    for metadata in results["metadatas"][0]
]

print("Expected:", repr(expected_section))
print("Retrieved:", [repr(section) for section in retrieved_sections])
print("Exact match:", expected_section in retrieved_sections)

Expected: '4.8. Defining Functions'
Retrieved: ["'4.8.3. Special parameters'", "'4.7. Defining Functions'", "'9.3.4. Method Objects'", "'4.7. Defining Functions'", "'4.8.1. Default Argument Values'"]
Exact match: False


In [50]:
# Re-run the retrieval evaluation after correcting the expected section.
evaluation_results = []

for item in evaluation_questions:
    results = retrieve_documents_v2(item["question"], top_k=5)

    retrieved_sections = [
        metadata["section"]
        for metadata in results["metadatas"][0]
    ]

    top_1_section = retrieved_sections[0]

    retrieved_correctly = (
        item["expected_section"] in retrieved_sections
    )

    evaluation_results.append({
        "question": item["question"],
        "expected_section": item["expected_section"],
        "top_1_section": top_1_section,
        "top_5_sections": " | ".join(retrieved_sections),
        "retrieved_correctly": retrieved_correctly
    })

evaluation_df = pd.DataFrame(evaluation_results)

evaluation_df

,question,expected_section,top_1_section,top_5_sections,retrieved_correctly
0,What is a Python list?,5.1. More on Lists,5.1. More on Lists,5.1. More on Lists | 5.5. Dictionaries | 3.1.3...,True
1,How does a for loop work in Python?,4.2. for Statements,9.8. Iterators,9.8. Iterators | 4.2. for Statements | 5.1.3. ...,True
2,What is a Python dictionary?,5.5. Dictionaries,5.5. Dictionaries,5.5. Dictionaries | 13. What Now? | | 1. Whet...,True
3,How do you handle exceptions in Python?,8.3. Handling Exceptions,8.3. Handling Exceptions,8.3. Handling Exceptions | 8.3. Handling Excep...,True
4,What is a Python class?,9. Classes,9. Classes,9. Classes | 9.3.2. Class Objects | 9.2. Pytho...,True
5,How do you define a function in Python?,4.8. Defining Functions,4.8.3. Special parameters,4.8.3. Special parameters | 4.7. Defining Func...,False
6,How can you read and write files in Python?,7.2. Reading and Writing Files,7.2.1. Methods of File Objects,7.2.1. Methods of File Objects | 7.2. Reading ...,True
7,What are list comprehensions in Python?,5.1.3. List Comprehensions,5.1.3. List Comprehensions,5.1.3. List Comprehensions | 5.5. Dictionaries...,True
8,What is a virtual environment in Python?,12.2. Creating Virtual Environments,12.2. Creating Virtual Environments,12.2. Creating Virtual Environments | 12.2. Cr...,True
9,How does Python import modules?,6.1. More on Modules,6.1.2. The Module Search Path,6.1.2. The Module Search Path | 6.1. More on M...,True


In [51]:
# Calculate the updated top-5 retrieval accuracy.
retrieval_accuracy = evaluation_df["retrieved_correctly"].mean()

print(f"Retrieval Accuracy (Top-5): {retrieval_accuracy:.2%}")

Retrieval Accuracy (Top-5): 90.00%


In [52]:
# Display all evaluation questions that failed to retrieve the expected section.
failed_retrievals = evaluation_df[
    evaluation_df["retrieved_correctly"] == False
]

failed_retrievals[
    [
        "question",
        "expected_section",
        "top_1_section",
        "top_5_sections"
    ]
]

,question,expected_section,top_1_section,top_5_sections
5,How do you define a function in Python?,4.8. Defining Functions,4.8.3. Special parameters,4.8.3. Special parameters | 4.7. Defining Func...


In [53]:
# Print the exact expected and retrieved section names for each failed question.
for _, row in failed_retrievals.iterrows():
    print("Question:", row["question"])
    print("Expected:", repr(row["expected_section"]))
    print("Retrieved:", row["top_5_sections"])
    print("-" * 80)

Question: How do you define a function in Python?
Expected: '4.8. Defining Functions'
Retrieved: 4.8.3. Special parameters | 4.7. Defining Functions | 9.3.4. Method Objects | 4.7. Defining Functions | 4.8.1. Default Argument Values
--------------------------------------------------------------------------------


In [54]:
# Recreate the evaluation dataset with the correct Python Tutorial section names.
evaluation_questions = [
    {
        "question": "What is a Python list?",
        "expected_section": "5.1. More on Lists"
    },
    {
        "question": "How does a for loop work in Python?",
        "expected_section": "4.2. for Statements"
    },
    {
        "question": "What is a Python dictionary?",
        "expected_section": "5.5. Dictionaries"
    },
    {
        "question": "How do you handle exceptions in Python?",
        "expected_section": "8.3. Handling Exceptions"
    },
    {
        "question": "What is a Python class?",
        "expected_section": "9. Classes"
    },
    {
        "question": "How do you define a function in Python?",
        "expected_section": "4.7. Defining Functions"
    },
    {
        "question": "How can you read and write files in Python?",
        "expected_section": "7.2. Reading and Writing Files"
    },
    {
        "question": "What are list comprehensions in Python?",
        "expected_section": "5.1.3. List Comprehensions"
    },
    {
        "question": "What is a virtual environment in Python?",
        "expected_section": "12.2. Creating Virtual Environments"
    },
    {
        "question": "How does Python import modules?",
        "expected_section": "6.1. More on Modules"
    }
]

# Verify the corrected function-definition evaluation entry.
for item in evaluation_questions:
    if "define a function" in item["question"].lower():
        print(item)

{'question': 'How do you define a function in Python?', 'expected_section': '4.7. Defining Functions'}


In [55]:
# Run the complete retrieval evaluation using the corrected expected sections.
evaluation_results = []

for item in evaluation_questions:
    results = retrieve_documents_v2(item["question"], top_k=5)

    retrieved_sections = [
        metadata["section"]
        for metadata in results["metadatas"][0]
    ]

    top_1_section = retrieved_sections[0]

    retrieved_correctly = (
        item["expected_section"] in retrieved_sections
    )

    evaluation_results.append({
        "question": item["question"],
        "expected_section": item["expected_section"],
        "top_1_section": top_1_section,
        "top_5_sections": " | ".join(retrieved_sections),
        "retrieved_correctly": retrieved_correctly
    })

evaluation_df = pd.DataFrame(evaluation_results)

evaluation_df

,question,expected_section,top_1_section,top_5_sections,retrieved_correctly
0,What is a Python list?,5.1. More on Lists,5.1. More on Lists,5.1. More on Lists | 5.5. Dictionaries | 3.1.3...,True
1,How does a for loop work in Python?,4.2. for Statements,9.8. Iterators,9.8. Iterators | 4.2. for Statements | 5.1.3. ...,True
2,What is a Python dictionary?,5.5. Dictionaries,5.5. Dictionaries,5.5. Dictionaries | 13. What Now? | | 1. Whet...,True
3,How do you handle exceptions in Python?,8.3. Handling Exceptions,8.3. Handling Exceptions,8.3. Handling Exceptions | 8.3. Handling Excep...,True
4,What is a Python class?,9. Classes,9. Classes,9. Classes | 9.3.2. Class Objects | 9.2. Pytho...,True
5,How do you define a function in Python?,4.7. Defining Functions,4.8.3. Special parameters,4.8.3. Special parameters | 4.7. Defining Func...,True
6,How can you read and write files in Python?,7.2. Reading and Writing Files,7.2.1. Methods of File Objects,7.2.1. Methods of File Objects | 7.2. Reading ...,True
7,What are list comprehensions in Python?,5.1.3. List Comprehensions,5.1.3. List Comprehensions,5.1.3. List Comprehensions | 5.5. Dictionaries...,True
8,What is a virtual environment in Python?,12.2. Creating Virtual Environments,12.2. Creating Virtual Environments,12.2. Creating Virtual Environments | 12.2. Cr...,True
9,How does Python import modules?,6.1. More on Modules,6.1.2. The Module Search Path,6.1.2. The Module Search Path | 6.1. More on M...,True


In [56]:
# Calculate the final top-5 retrieval accuracy for the 10-question evaluation set.
retrieval_accuracy = evaluation_df["retrieved_correctly"].mean()

print(f"Retrieval Accuracy (Top-5): {retrieval_accuracy:.2%}")

Retrieval Accuracy (Top-5): 100.00%


In [57]:
# Save the retrieval evaluation results to a CSV file for later reporting.
evaluation_df.to_csv(
    "/content/retrieval_evaluation.csv",
    index=False
)

print("Saved retrieval evaluation to /content/retrieval_evaluation.csv")

Saved retrieval evaluation to /content/retrieval_evaluation.csv


In [58]:
# Display the final retrieval accuracy and number of evaluated questions.
print(f"Number of evaluation questions: {len(evaluation_df)}")
print(f"Retrieval Accuracy (Top-5): {retrieval_accuracy:.2%}")

Number of evaluation questions: 10
Retrieval Accuracy (Top-5): 100.00%


In [59]:
# Export the RAG configuration so the backend can reuse the same pipeline settings.
import json

with open("/content/rag_config.json", "w", encoding="utf-8") as file:
    json.dump(RAG_CONFIG, file, indent=4)

print("Saved RAG configuration to /content/rag_config.json")

Saved RAG configuration to /content/rag_config.json


In [60]:
# Package the persisted ChromaDB vector store for transfer to the local project.
!zip -qr /content/chroma_db_v2.zip /content/chroma_db_v2

print("Created /content/chroma_db_v2.zip")

Created /content/chroma_db_v2.zip
